# 04 · 子查询与集合操作

> 前面的 SQL 都是“一条查询”。真实分析里你常需要“先查一个中间结果，再拿它去查外层”——这就是子查询。
> 你在 Pandas 里会先算一个临时 DataFrame 再 `merge` / `filter`，SQL 的子查询就是这个“临时结果”，只不过它是**写在另一条 SQL 内部的查询**。这一课是“从只会写平铺 SQL，到能搭出复杂分析”的关键一跃，也为下一课窗口函数铺路。

## 本课地图
1. 什么是子查询：按返回形状分类（标量 / 列 / 行）
2. IN 与相关子查询（EXISTS）
3. 派生表：子查询放在 FROM 里
4. 集合操作：UNION / UNION ALL / INTERSECT / EXCEPT
5. 用子查询硬写“组内 TopN”（然后你就会理解窗口函数为何是福音）

In [1]:
# 统一环境：连接数据库（绝对路径，可反复运行）
import sqlite3
import pandas as pd

# 两个库都用绝对路径写死，避免“相对路径在哪个工作目录下”的坑
SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
SS_DB  = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\superstore.db"

# 让 DataFrame 打印好看一点（你已经熟悉 Pandas，这里直接拿它当 SQL 的“显示层”）
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# 教学默认连 shop.db；需要 superstore 时再切
conn = sqlite3.connect(SHOP_DB)

def q(sql, conn=conn):
    """执行一条 SQL，把结果表直接返回成 Pandas DataFrame。"""
    return pd.read_sql_query(sql, conn)

print("已连接 shop.db。可用 q('SELECT ...') 跑查询。")
print("shop.db 表：", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

已连接 shop.db。可用 q('SELECT ...') 跑查询。
shop.db 表： ['customers', 'employees', 'order_items', 'orders', 'products']


## 1. 子查询的三种返回形状

子查询就是“查询里套查询”，用括号包起来。按它**返回几行几列**，用法完全不同：

| 子查询返回 | 能放在哪 | 类比 |
|---|---|---|
| **标量**（1 行 1 列） | 表达式位置，能当一个值用 | Python 里 `f(g())` |
| **一列**（多行 1 列） | `IN (...)` 里 | 一个 list |
| **一张表**（多行多列） | `FROM (...)` 里（派生表） | 一个临时 DataFrame |

In [2]:
# (1) 标量子查询：返回“全库平均单笔销售额”一个数，直接拿去比
q('''
SELECT item_id, order_id, sales
FROM order_items
WHERE sales > (SELECT AVG(sales) FROM order_items)   -- 括号里算出一个值
ORDER BY sales DESC
LIMIT 8
''')

,item_id,order_id,sales
0,1,1,43202.52
1,403,181,43202.52
2,465,205,43202.52
3,479,212,43202.52
4,946,396,43202.52
5,1138,469,43202.52
6,2591,1048,43202.52
7,2618,1059,43202.52


In [3]:
# (2) 列子查询：返回“华东地区所有客户 id”这一列，用 IN 包住
q('''
SELECT order_id, customer_id, order_date
FROM orders
WHERE customer_id IN (
        SELECT customer_id FROM customers WHERE region = '华东'
      )
LIMIT 8
''')

,order_id,customer_id,order_date
0,82,1,2025-04-02
1,212,1,2023-07-02
2,452,1,2025-12-14
3,641,1,2022-06-06
4,684,1,2022-10-03
5,1439,1,2023-07-15
6,1584,1,2022-01-08
7,2064,1,2024-09-27


> 标量子查询是最自然的：它就是“先算出一个基准数，再跟每行比”。列子查询配 `IN`，相当于“这个值在不在那个名单里”。

## 2. IN vs EXISTS，以及相关子查询

### 非相关子查询
上面那个 `IN (SELECT ...)` 是**非相关**的：内层只算一次，算出一个客户 id 列表，外层再用。内外层互不依赖。

### 相关子查询
内层查询**引用了外层的列**，于是外层每拿一行，内层就要重新算一遍。

In [4]:
# EXISTS：“有没有任何一笔订单属于这个客户”
# 内层 o.customer_id = c.customer_id 引用了外层 -> 相关子查询
q('''
SELECT c.customer_id, c.name, c.region
FROM customers c
WHERE EXISTS (
        SELECT 1 FROM orders o
        WHERE o.customer_id = c.customer_id
      )
LIMIT 8
''')

,customer_id,name,region
0,1,许军,华东
1,2,马欣怡,华东
2,3,郭秀英,华北
3,4,梁艳,华南
4,5,何霞,华北
5,6,郭杰,西南
6,7,胡思远,华南
7,8,孙娜,华中


`EXISTS` 只关心“子查询能不能返回行”，能就真、不能就假，**不关心返回的具体列**（所以大家习惯写 `SELECT 1`）。

**IN vs EXISTS 怎么选？** 经验：
- 子查询结果集**小**、外表大 → `IN` 往往更快；
- 子查询结果集**大**、外表小 → `EXISTS`（找到一条匹配就停）往往更快；
- 现代优化器会自动改写，入门阶段先把两者都读懂，性能调优是后话。

## 3. 派生表：把子查询放在 FROM 里

把子查询包起来当成一张“临时表”，放在 `FROM (...)` 里，外面再聚合。这是**分步分析**的核心手段：先算中间指标，再汇总。

In [5]:
# 第一步（内层）：算出每个客户的总消费额；
# 第二步（外层）：在这些客户里挑消费额前 10
q('''
SELECT t.customer_id, t.name, t.total_spent
FROM (
    SELECT c.customer_id, c.name, SUM(oi.sales) AS total_spent
    FROM customers c
    JOIN orders o       ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id, c.name
) t
ORDER BY t.total_spent DESC
LIMIT 10
''')

,customer_id,name,total_spent
0,239,李超,278713.38
1,253,林梓涵,219832.72
2,160,马平,213901.28
3,51,梁一诺,208011.66
4,149,林欣怡,195391.93
5,123,何涛,192931.33
6,242,罗秀英,181633.85
7,97,马刚,179549.14
8,166,吴超,179051.56
9,225,何梓涵,176793.62


> 派生表必须起别名（这里是 `t`），否则数据库不知道怎么引用它。这跟 Pandas 里“先 groupby 出一个临时 df，再对它排序取 head”一模一样。

## 4. 集合操作：把多个查询的结果“叠”起来

前面都是“横向拼列”(JOIN)。集合操作是“纵向堆行”，要求两边**列数相同、类型兼容**：

- `UNION`：合并，并**去重**；
- `UNION ALL`：合并，**不去重**（快得多，能用就别用 UNION）；
- `INTERSECT`：取交集；
- `EXCEPT`：取差集（在 A 不在 B）。

In [6]:
# UNION ALL：把“员工姓名”和“客户姓名”纵向堆成一张大名单
# （结果 54+300 行，DataFrame 会自动截断显示前若干行）
q('''
SELECT name FROM employees
UNION ALL
SELECT name FROM customers
''')

,name
0,张总
1,销售总监
2,华东销售经理
3,华北销售经理
4,华南销售经理
...,...
349,许明
350,谢霞
351,朱芳
352,张伟


In [7]:
# UNION（去重）vs UNION ALL：列出来自哪些来源
q('''
SELECT '员工' AS 来源, name FROM employees WHERE name LIKE '张%'
UNION
SELECT '客户' AS 来源, name FROM customers WHERE name LIKE '张%'
ORDER BY name
LIMIT 10
''')

,来源,name
0,客户,张一诺
1,客户,张伟
2,客户,张娜
3,客户,张思远
4,员工,张总
5,员工,张杰
6,客户,张梓涵
7,客户,张洋
8,客户,张涛
9,客户,张秀英


In [8]:
# EXCEPT：哪些地区出现在 customers 里，却从没在 orders 里出现？
# （本库地区基本重合，结果可能为空——理解它的语义即可）
q('''
SELECT region FROM customers
EXCEPT
SELECT region FROM orders
''')

,region


> 细节：`UNION` 默认排序去重，大数据量慢；能确定没有重复就用 `UNION ALL`。集合操作只看“对应位置的列”，列名以第一个 SELECT 为准。

## 5. 用子查询硬写“组内 TopN”（为窗口函数铺垫）

需求：**每个销售大区里，销售额最高的那个商品子类别**。没有窗口函数时，我们只能用相关子查询“硬凑”。

In [9]:
# 先看：每个地区每个子类别的总销售额（这是我们的“分组聚合表”）
q('''
SELECT o.region, p.subcategory, SUM(oi.sales) AS sub_sales
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p     ON p.product_id = oi.product_id
GROUP BY o.region, p.subcategory
ORDER BY o.region, sub_sales DESC
LIMIT 15
''')

,region,subcategory,sub_sales
0,华东,电脑,1739774.80
1,华东,手机,1190876.84
2,华东,平板,865685.17
3,华东,沙发,717605.22
4,华东,显示器,460910.21
5,华东,床,430855.13
6,华东,桌子,263085.79
7,华东,耳机,119879.51
8,华东,书架,118562.26
9,华东,椅子,89272.31


In [10]:
# 硬写组内 Top1：用相关子查询找“比我这个组里其它行都大的那一行”
# 翻译：选出 (地区, 子类别)，且不存在另一个同地区的子类别销售额比它还高
q('''
SELECT a.region, a.subcategory, a.sub_sales
FROM (
    SELECT o.region, p.subcategory, SUM(oi.sales) AS sub_sales
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p     ON p.product_id = oi.product_id
    GROUP BY o.region, p.subcategory
) a
WHERE NOT EXISTS (
    SELECT 1 FROM (
        SELECT o.region, p.subcategory, SUM(oi.sales) AS sub_sales
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.order_id
        JOIN products p     ON p.product_id = oi.product_id
        GROUP BY o.region, p.subcategory
    ) b
    WHERE b.region = a.region AND b.sub_sales > a.sub_sales
)
ORDER BY a.region
''')

,region,subcategory,sub_sales
0,华东,电脑,1739774.80
1,华中,电脑,1533092.56
2,华北,电脑,1080601.65
3,华南,电脑,1781960.05
4,西南,手机,933437.72


> 感觉到别扭了吧？同一段聚合 SQL 复制粘贴了两遍，只为了“自比”。这就是为什么下一课的窗口函数是刚需——它能用 `RANK() OVER (PARTITION BY region ORDER BY sub_sales DESC)` 一行搞定，既不用重复子查询，也不会漏边界。**你现在有多痛，下一课学窗口函数就有多爽。**

## 动手小练习

1. 用标量子查询：列出销售额高于全库订单明细中位数附近、你能想到的对比（比如高于平均）的明细。
2. 用 `IN` 子查询：找出“至少买过一次科技产品”的客户 id。
3. 用派生表：算出每个客户的下单次数，再筛出“下单次数 >= 5”的客户。

In [11]:
# TODO(你): 在下面练习
# q(''' ... ''')

## 本课小结

- 子查询按返回形状分：标量（当值）、一列（配 IN）、一张表（当派生表）。
- 非相关子查询只算一次；相关子查询引用外层列，外层每行算一次。
- EXISTS 判断“有没有匹配行”，不看具体列；IN/EXISTS 选择看哪边集合小。
- 派生表 = FROM 里的临时表，必须起别名，是分步分析的核心。
- UNION ALL 堆行不去重；UNION 去重慢；INTERSECT 交、EXCEPT 差。
- 组内 TopN 用子查询写又长又重复——下一课窗口函数就是来救场的。

## 来源与延伸

- Mode SQL Tutorial · 子查询 (Subqueries)：https://mode.com/sql-tutorial/sql-sub-queries/
- SQLBolt 第 13~14 课（子查询）：https://sqlbolt.com/
- SQLite 官方：表达式 / 子查询与集合操作：https://www.sqlite.org/lang_expr.html
- LeetCode Database（中等/困难大量考相关子查询与 EXISTS）：https://leetcode.com/problemset/database/
- DataLemur（面试高频的“组内 TopN”题在这里先睹为快）：https://datalemur.com/